# ETL — Dimensión Hashtag (`dim_hashtag`)

Este notebook extrae los hashtags únicos desde la capa Silver (`tiktok_data_eng.silver.silver_tiktok`),
calcula su longitud en caracteres y ejecuta un `MERGE` idempotente en `tiktok_data_eng.gold.dim_hashtag`
generando la clave subrogada incremental `hashtag_id`.

In [0]:
%sql
-- Inserción / Actualización idempotente (MERGE) en dim_hashtag utilizando CTE

WITH hashtags_unicos AS (
    SELECT DISTINCT 
        TRIM(hashtag) AS hashtag
    FROM tiktok_data_eng.silver.silver_tiktok
    WHERE hashtag IS NOT NULL AND TRIM(hashtag) <> ''
)
MERGE INTO tiktok_data_eng.gold.dim_hashtag AS target
USING (
    SELECT 
        hashtag,
        LENGTH(hashtag) AS longitud
    FROM hashtags_unicos
) AS source
ON target.hashtag = source.hashtag
WHEN NOT MATCHED THEN INSERT (
    hashtag,
    longitud,
    _created_at
) VALUES (
    source.hashtag,
    source.longitud,
    CURRENT_TIMESTAMP()
);

In [0]:
%sql
-- Validación de calidad y volumetría en dim_hashtag
SELECT 
    COUNT(*) AS total_filas,
    COUNT(DISTINCT hashtag_id) AS total_ids_subrogados,
    COUNT(DISTINCT hashtag) AS total_hashtags_unicos,
    SUM(CASE WHEN hashtag_id IS NULL THEN 1 ELSE 0 END) AS nulos_pk,
    COUNT(*) - COUNT(DISTINCT hashtag) AS duplicados
FROM tiktok_data_eng.gold.dim_hashtag;